# Actividad 6: Torre de Hanoi con Búsqueda A* usando una heurística desarrollada por uno mismo 

## Objetivo

Resolver la Torre de Hanoi utilizando **A***, un algoritmo distinto de BFS, con una heurística desarrollada para este problema.

- Cada estado representa las tres torres y sus discos.
- Cada movimiento legal tiene costo 1.
- La función de evaluación es $f(n) = g(n) + h(n)$.
- La heurística propia cuenta cuántos discos todavía no están en la torre objetivo.

La heurística es admisible porque cada disco que no está en su torre final necesita realizar al menos un movimiento.

In [1]:
from heapq import heappop, heappush
from itertools import count
from math import inf


def estado_inicial(numero_discos):
    """Crea el estado inicial: todos los discos están en la torre 0."""
    return (tuple(range(numero_discos, 0, -1)), (), ())


def estado_objetivo(numero_discos):
    """Crea el estado objetivo: todos los discos están en la torre 2."""
    return ((), (), tuple(range(numero_discos, 0, -1)))


def movimientos_legales(estado):
    """Genera estados sucesores aplicando únicamente movimientos válidos."""
    for origen in range(3):
        if not estado[origen]:
            continue

        disco = estado[origen][-1]
        for destino in range(3):
            if origen == destino:
                continue
            if estado[destino] and estado[destino][-1] < disco:
                continue

            nuevas_torres = [list(torre) for torre in estado]
            nuevas_torres[origen].pop()
            nuevas_torres[destino].append(disco)
            nuevo_estado = tuple(tuple(torre) for torre in nuevas_torres)
            yield nuevo_estado, (disco, origen, destino)


def heuristica(estado, numero_discos):
    """Heurística propia: cantidad de discos que no están en la torre objetivo."""
    torre_objetivo = estado[2]
    return sum(disco not in torre_objetivo for disco in range(1, numero_discos + 1))


def reconstruir_camino(predecesor, estado_final):
    """Reconstruye la secuencia de movimientos desde el inicio hasta la meta."""
    camino = []
    estado_actual = estado_final

    while predecesor[estado_actual][0] is not None:
        estado_anterior, movimiento = predecesor[estado_actual]
        camino.append(movimiento)
        estado_actual = estado_anterior

    camino.reverse()
    return camino


def resolver_hanoi_a_estrella(numero_discos):
    """Resuelve Torre de Hanoi con A* y devuelve camino y métricas."""
    inicio = estado_inicial(numero_discos)
    objetivo = estado_objetivo(numero_discos)
    contador = count()

    costos = {inicio: 0}
    predecesor = {inicio: (None, None)}
    frontera = [(heuristica(inicio, numero_discos), 0, next(contador), inicio)]
    estados_explorados = 0

    while frontera:
        _, costo_actual, _, estado_actual = heappop(frontera)

        if costo_actual != costos[estado_actual]:
            continue

        estados_explorados += 1
        if estado_actual == objetivo:
            camino = reconstruir_camino(predecesor, estado_actual)
            return camino, estados_explorados

        for nuevo_estado, movimiento in movimientos_legales(estado_actual):
            nuevo_costo = costo_actual + 1
            if nuevo_costo < costos.get(nuevo_estado, inf):
                costos[nuevo_estado] = nuevo_costo
                predecesor[nuevo_estado] = (estado_actual, movimiento)
                prioridad = nuevo_costo + heuristica(nuevo_estado, numero_discos)
                heappush(frontera, (prioridad, nuevo_costo, next(contador), nuevo_estado))

    raise ValueError('No se encontró una solución para la instancia indicada.')

In [2]:
def aplicar_movimiento(estado, movimiento):
    """Aplica un movimiento y verifica que respete las reglas de Hanoi."""
    disco, origen, destino = movimiento
    assert estado[origen] and estado[origen][-1] == disco
    assert not estado[destino] or estado[destino][-1] > disco

    nuevas_torres = [list(torre) for torre in estado]
    nuevas_torres[origen].pop()
    nuevas_torres[destino].append(disco)
    return tuple(tuple(torre) for torre in nuevas_torres)


def verificar_solucion(numero_discos, camino):
    """Comprueba legalidad, estado final y cantidad óptima de movimientos."""
    estado = estado_inicial(numero_discos)
    for movimiento in camino:
        estado = aplicar_movimiento(estado, movimiento)

    assert estado == estado_objetivo(numero_discos)
    assert len(camino) == 2 ** numero_discos - 1
    return True

In [3]:
for numero_discos in [3, 4]:
    camino, estados_explorados = resolver_hanoi_a_estrella(numero_discos)
    verificar_solucion(numero_discos, camino)

    print(f'\nTorre de Hanoi con {numero_discos} discos')
    print(f'Movimientos de la solución: {len(camino)}')
    print(f'Movimientos óptimos esperados: {2 ** numero_discos - 1}')
    print(f'Estados explorados: {estados_explorados}')
    print('Solución verificada: correcta')

print('\nPrimeros movimientos para 3 discos:')
for numero, (disco, origen, destino) in enumerate(resolver_hanoi_a_estrella(3)[0], start=1):
    print(f'{numero}. Mover disco {disco} de torre {origen + 1} a torre {destino + 1}')


Torre de Hanoi con 3 discos
Movimientos de la solución: 7
Movimientos óptimos esperados: 7
Estados explorados: 18
Solución verificada: correcta

Torre de Hanoi con 4 discos
Movimientos de la solución: 15
Movimientos óptimos esperados: 15
Estados explorados: 54
Solución verificada: correcta

Primeros movimientos para 3 discos:
1. Mover disco 1 de torre 1 a torre 3
2. Mover disco 2 de torre 1 a torre 2
3. Mover disco 1 de torre 3 a torre 2
4. Mover disco 3 de torre 1 a torre 3
5. Mover disco 1 de torre 2 a torre 1
6. Mover disco 2 de torre 2 a torre 3
7. Mover disco 1 de torre 1 a torre 3
